# Task 4 — Health Chatbot (Gemini 2.5 Flash)

## Objective
Create a safe, factual health-information chatbot powered by **Gemini 2.5 Flash** (free tier via `google-generativeai`).

## Dataset
No dataset is required — this is a conversational assistant.

## Approach
- Use a strict system prompt to avoid diagnosis/prescriptions
- Implement a lightweight safety filter to refuse: self-harm, prescription dosages, emergencies
- Provide an interactive CLI loop (`input()`), exit on `quit`
- Handle missing API key and network/API errors gracefully

## Final Summary
This notebook demonstrates how to build a basic health chatbot using Gemini with explicit safety rules and refusal handling.

In [ ]:
# pip install google-generativeai

import os
import re
from typing import Optional

try:
    import google.generativeai as genai
except Exception as e:
    genai = None
    print('Failed to import google-generativeai. Install it with pip. Error:', e)


In [ ]:
SYSTEM_PROMPT = (
    'You are a helpful medical assistant. Provide clear, factual, safe information. '
    'Never diagnose or prescribe. Recommend consulting a doctor for serious symptoms.'
)

UNSAFE_PATTERNS = [
    r'\b(suicide|kill myself|self-harm|hurt myself)\b',
    r'\b(overdose|od)\b',
    r'\b(what dose|dosage|mg|milligram|how many pills)\b',
    r'\b(chest pain|can\s*\'?t breathe|difficulty breathing|severe bleeding|stroke|heart attack)\b',
    r'\b(emergency|call 911|ambulance)\b',
]

def is_unsafe(user_text: str) -> bool:
    text = (user_text or '').lower()
    return any(re.search(p, text) for p in UNSAFE_PATTERNS)

def refusal_message() -> str:
    return (
        'I\'m sorry, but I can\'t help with that. '
        'If this is an emergency or you might be in danger, please contact local emergency services or a medical professional right now.'
    )


In [ ]:
def build_gemini_model() -> Optional[object]:
    if genai is None:
        return None

    api_key = os.getenv('GOOGLE_API_KEY')
    if not api_key:
        print('Missing GOOGLE_API_KEY env var. Set it to your free Google AI Studio key.')
        return None

    try:
        genai.configure(api_key=api_key)
        model = genai.GenerativeModel(
            model_name='gemini-2.5-flash',
            system_instruction=SYSTEM_PROMPT,
        )
        return model
    except Exception as e:
        print('Failed to configure Gemini client:', e)
        return None

model = build_gemini_model()
model

## Examples (safe vs unsafe)

**Safe**
- "What are common causes of a mild headache?"
- "How can I reduce a sore throat at home?"

**Unsafe (will be refused)**
- "How many mg of ibuprofen should I take?"
- "I have chest pain and can’t breathe — what do I do?"
- "I want to hurt myself"

In [ ]:
def ask_gemini(user_text: str) -> str:
    if is_unsafe(user_text):
        return refusal_message()

    if model is None:
        return (
            'Gemini model is not available. Ensure `google-generativeai` is installed and '
            'the `GOOGLE_API_KEY` environment variable is set.'
        )

    try:
        resp = model.generate_content(user_text)
        return (resp.text or '').strip() or 'No response text received.'
    except Exception as e:
        return f'API request failed (network/auth/quota issue): {e}'

# Quick demo (optional):
print('Safe example ->')
print(ask_gemini('What are common causes of mild headaches?'))
print('\nUnsafe example ->')
print(ask_gemini('How many mg of ibuprofen should I take?'))

In [ ]:
def run_cli():
    print('Health Chatbot (Gemini 2.5 Flash). Type `quit` to exit.')
    while True:
        user_text = input('You: ').strip()
        if not user_text:
            continue
        if user_text.lower() == 'quit':
            print('Bye!')
            break
        print('Bot:', ask_gemini(user_text))

# Uncomment to run interactively in a local Jupyter environment:
# run_cli()
